[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/information_theory/05_mutual_information/exercises.ipynb)

# Module 05 — Exercises: Mutual Information

Twenty-eight solved problems in four tiers. Every problem carries a statement, a one-line intuition,
a stepwise solution, a boxed answer, a key takeaway, and — wherever the answer is numeric or
algorithmic — a code cell that recomputes it and prints the check.

Theorem, proof and example numbers refer to
[first_principles.ipynb](first_principles.ipynb). Symbols follow
[the notation register](../../docs/notation.md): $H(X, Y)$ is joint entropy, KL divergence is
written with `\parallel`, the InfoNCE loss is **unnormalized**, and every numeric answer carries its
unit — bits for $\log_2$, nats for $\ln$.

The preamble below is shared by every code cell in this notebook.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

EPS = np.finfo(float).eps
LOG2 = np.log(2.0)


def entropy_bits(p):
    """Shannon entropy of a probability array, in bits."""
    p = np.asarray(p, dtype=float).ravel()
    p = p[p > 0.0]
    return float(-(p * np.log2(p)).sum())


def binary_entropy(p):
    """H_b(p) in bits, safe at p = 0 and p = 1."""
    p = np.asarray(p, dtype=float)
    out = np.zeros(p.shape, dtype=float)
    m = (p > 0.0) & (p < 1.0)
    q = p[m]
    out[m] = -q * np.log2(q) - (1.0 - q) * np.log2(1.0 - q)
    return out if out.shape else float(out)


def mutual_information_bits(P):
    """I(X; Y) in bits from a joint probability table P[x, y]."""
    P = np.asarray(P, dtype=float)
    return entropy_bits(P.sum(axis=1)) + entropy_bits(P.sum(axis=0)) - entropy_bits(P)


def conditional_mi_bits(P):
    """I(X; Y | Z) in bits from a joint table P[x, y, z]."""
    P = np.asarray(P, dtype=float)
    total = 0.0
    for k in range(P.shape[2]):
        pz = P[:, :, k].sum()
        if pz > 0.0:
            total += pz * mutual_information_bits(P[:, :, k] / pz)
    return total


print(f"machine epsilon = {EPS:.4e}")
print(f"1 nat           = {1.0 / LOG2:.6f} bits")

machine epsilon = 2.2204e-16
1 nat           = 1.442695 bits


## L0 — Concept Checks

### Problem L0.1 — Independent variables carry zero information

**Statement.** Two fair coins are tossed independently. Compute $I(X; Y)$ from Definition 3.1 and
from the entropy form of Theorem 4.1.

**Intuition.** If the joint law is the product of the marginals, every log-ratio in the sum is
$\log 1$.

**Solution.**

*Step 1.* Independence gives $p(x, y) = p(x)p(y) = \tfrac{1}{4}$ in all four cells, so every ratio
inside the logarithm is $1$ and every term is zero.

*Step 2.* From the entropy form: $H(X) = 1$ bit and $Y$ says nothing, so $H(X \mid Y) = 1$ bit and
$I = 1 - 1 = 0$.

$$
\boxed{I(X; Y) = 0 \text{ exactly, because } P_{XY} = P_X P_Y}
$$

**Key takeaway.** Zero is not an approximation here — it is the equality case of Theorem 4.2.

In [2]:
P = np.full((2, 2), 0.25)
print("joint table:\n", P)
print(f"I(X;Y) = {mutual_information_bits(P):.3e} bits")
assert abs(mutual_information_bits(P)) < 1e-15

joint table:
 [[0.25 0.25]
 [0.25 0.25]]
I(X;Y) = 0.000e+00 bits


### Problem L0.2 — Entropy is self-information

**Statement.** Show that $I(X; X) = H(X)$ for a discrete variable, and say why the analogous
statement fails for a continuous one.

**Intuition.** Knowing $X$ determines $X$, so nothing is left over.

**Solution.**

*Step 1.* $H(X \mid X) = 0$, so Theorem 4.1 gives $I(X; X) = H(X) - 0 = H(X)$.

*Step 2.* For a continuous $X$ the conditional differential entropy $h(X \mid X)$ is $-\infty$, so
$I(X; X) = +\infty$: a real number carries unboundedly many bits, and only quantization or noise
makes the quantity finite.

$$
\boxed{I(X; X) = H(X) \text{ (discrete)}, \qquad I(X; X) = +\infty \text{ (continuous)}}
$$

**Key takeaway.** Entropy is the diagonal of mutual information; the infinite continuous value is
the same phenomenon as $I \to \infty$ when $\lvert \rho \rvert \to 1$ in Theorem 4.6.

In [3]:
p = np.array([0.5, 0.3, 0.2])
P_diag = np.diag(p)
print(f"H(X)     = {entropy_bits(p):.10f} bits")
print(f"I(X;X)   = {mutual_information_bits(P_diag):.10f} bits")
assert abs(mutual_information_bits(P_diag) - entropy_bits(p)) < 1e-14

H(X)     = 1.4854752972 bits
I(X;X)   = 1.4854752972 bits


### Problem L0.3 — Zero correlation, positive information

**Statement.** Let $X$ be uniform on $\lbrace -1, 0, 1 \rbrace$ and $Y = X^2$. Compute
$\operatorname{Corr}(X, Y)$ and $I(X; Y)$ in bits.

**Intuition.** Correlation sees only a straight line, and $X^2$ is symmetric about zero.

**Solution.**

*Step 1.* $\mathbb{E}[X] = 0$ and $\mathbb{E}[XY] = \mathbb{E}[X^3] = 0$ by symmetry, so
$\operatorname{Cov}(X, Y) = 0$ and $\operatorname{Corr}(X, Y) = 0$.

*Step 2.* $Y = 0$ with probability $\tfrac{1}{3}$ and $Y = 1$ with probability $\tfrac{2}{3}$, so
$H(Y) = H_b(1/3) = 0.918296$ bits. Since $Y$ is a function of $X$, $H(Y \mid X) = 0$ and
$I(X; Y) = H(Y)$.

$$
\boxed{\operatorname{Corr}(X, Y) = 0 \quad \text{but} \quad I(X; Y) = H_b(1/3) = 0.918296 \text{ bits}}
$$

**Key takeaway.** Correlation is a second-moment summary; mutual information sees the whole joint
law, which is why it is the right screen when the functional form is unknown.

In [4]:
xs = np.array([-1.0, 0.0, 1.0])
px = np.full(3, 1.0 / 3.0)
P = np.zeros((3, 2))
P[0, 1] = P[1, 0] = P[2, 1] = 1.0 / 3.0
cov = float((px * xs * xs ** 2).sum() - (px * xs).sum() * (px * xs ** 2).sum())
print(f"Cov(X, Y) = {cov:.3e}")
print(f"I(X;Y)    = {mutual_information_bits(P):.6f} bits")
print(f"H_b(1/3)  = {binary_entropy(np.array(1.0 / 3.0)):.6f} bits")
assert abs(cov) < 1e-15
assert abs(mutual_information_bits(P) - binary_entropy(np.array(1.0 / 3.0))) < 1e-14

Cov(X, Y) = 0.000e+00
I(X;Y)    = 0.918296 bits
H_b(1/3)  = 0.918296 bits


### Problem L0.4 — Pointwise information can be negative

**Statement.** For the joint table of Example 6.1, $p(0,0) = p(1,1) = 0.4$ and
$p(0,1) = p(1,0) = 0.1$, compute the pointwise mutual information $i(0; 1)$ of Definition 3.2.

**Intuition.** The pair $(0, 1)$ occurs less often than chance would predict, so the log-ratio is
negative.

**Solution.**

*Step 1.* Both marginals are uniform, so $p(0)p(1) = 0.25$ while $p(0, 1) = 0.1$.

*Step 2.* $i(0; 1) = \log_2 \frac{0.1}{0.25} = \log_2 0.4 = -1.321928$ bits.

$$
\boxed{i(0; 1) = \log_2 0.4 = -1.321928 \text{ bits}}
$$

**Key takeaway.** Only the *average* is nonnegative (Theorem 4.2); individual pointwise terms are
free to be negative, which is exactly what "less likely than chance" looks like in bits.

In [5]:
P = np.array([[0.4, 0.1], [0.1, 0.4]])
pmi = np.log2(P / np.outer(P.sum(axis=1), P.sum(axis=0)))
print("pointwise mutual information table (bits):\n", pmi)
print(f"i(0;1) = {pmi[0, 1]:.6f} bits;  average I = {mutual_information_bits(P):.6f} bits")
assert pmi[0, 1] < 0.0 < mutual_information_bits(P)

pointwise mutual information table (bits):
 [[ 0.6781 -1.3219]
 [-1.3219  0.6781]]
i(0;1) = -1.321928 bits;  average I = 0.278072 bits


### Problem L0.5 — Nats, bits, and the $\log K$ ceiling

**Statement.** A contrastive run uses a batch of $K = 256$. State the ceiling of Theorem 4.8 in
nats and in bits.

**Intuition.** The ceiling is $\log K$ in whatever base the logarithm is taken.

**Solution.**

*Step 1.* In nats, $\ln 256 = 5.545177$.

*Step 2.* In bits, $\log_2 256 = 8$ exactly, since $256 = 2^8$. The two differ by the factor
$\ln 2 = 0.693147$.

$$
\boxed{\text{ceiling} = \ln 256 = 5.545177 \text{ nats} = \log_2 256 = 8 \text{ bits}}
$$

**Key takeaway.** A number with no unit is not an answer: $5.55$ and $8$ are the same ceiling, and
the notation register requires the unit to be attached.

In [6]:
K = 256
print(f"log_e {K} = {np.log(K):.6f} nats")
print(f"log_2 {K} = {np.log2(K):.6f} bits")
print(f"conversion: {np.log(K):.6f} / ln 2 = {np.log(K) / LOG2:.6f}")
assert abs(np.log(K) / LOG2 - 8.0) < 1e-12

log_e 256 = 5.545177 nats
log_2 256 = 8.000000 bits
conversion: 5.545177 / ln 2 = 8.000000


### Problem L0.6 — The smaller entropy is the ceiling

**Statement.** Show in one line that $I(X; Y) \le \min\left( H(X), H(Y) \right)$ for discrete
variables.

**Intuition.** Mutual information is an entropy minus something nonnegative.

**Solution.**

*Step 1.* Theorem 4.1 gives $I(X; Y) = H(X) - H(X \mid Y)$, and conditional entropy of a discrete
variable is nonnegative, so $I(X; Y) \le H(X)$.

*Step 2.* The same argument with the roles exchanged gives $I(X; Y) \le H(Y)$.

$$
\boxed{I(X; Y) \le \min\left( H(X), H(Y) \right), \text{ with equality iff one variable determines the other}}
$$

**Key takeaway.** A label with three bits of entropy caps what any feature can be worth, however
rich the feature is.

In [7]:
for trial in range(4):
    draw = rng.random((3, 4)) ** 2
    P = draw / draw.sum()
    I = mutual_information_bits(P)
    cap = min(entropy_bits(P.sum(axis=1)), entropy_bits(P.sum(axis=0)))
    print(f"  trial {trial}: I = {I:.6f} bits   min(H(X), H(Y)) = {cap:.6f} bits   ok: {I <= cap}")
    assert I <= cap + 1e-15

  trial 0: I = 0.260557 bits   min(H(X), H(Y)) = 1.361937 bits   ok: True
  trial 1: I = 0.499091 bits   min(H(X), H(Y)) = 1.563249 bits   ok: True
  trial 2: I = 0.233581 bits   min(H(X), H(Y)) = 1.471986 bits   ok: True
  trial 3: I = 0.244108 bits   min(H(X), H(Y)) = 1.509526 bits   ok: True


## L1 — Foundations

### Problem L1.1 — Mutual information from a joint table

**Statement.** The joint law of $(X, Y)$ on $\lbrace 0, 1 \rbrace^2$ is $p(0,0) = 0.4$,
$p(0,1) = 0.1$, $p(1,0) = 0.1$, $p(1,1) = 0.4$. Compute $H(X)$, $H(X \mid Y)$ and $I(X; Y)$ in bits,
and cross-check with the union form.

**Intuition.** The table is a noisy copy: $Y$ usually equals $X$, so it removes most but not all of
the uncertainty.

**Solution.**

*Step 1 — marginals.* Both rows and both columns sum to $0.5$, so $H(X) = H(Y) = 1$ bit.

*Step 2 — conditional.* Given $Y = 0$, which has probability $0.5$, the conditional law of $X$ is
$(0.8, 0.2)$; by symmetry the same at $Y = 1$. Hence

$$
H(X \mid Y) = H_b(0.2) = -0.8\log_2 0.8 - 0.2 \log_2 0.2 = 0.721928 \text{ bits}.
$$

*Step 3 — mutual information.* $I(X; Y) = 1 - 0.721928 = 0.278072$ bits.

*Step 4 — cross-check.* $H(X, Y) = -2(0.4\log_2 0.4) - 2(0.1\log_2 0.1) = 1.721928$, and the union
form gives $1 + 1 - 1.721928 = 0.278072$ bits.

$$
\boxed{H(X) = 1, \quad H(X \mid Y) = H_b(0.2) = 0.721928, \quad I(X; Y) = 0.278072 \text{ bits}}
$$

**Key takeaway.** This table is the binary symmetric channel of Example 6.4 with crossover $0.2$
driven by a uniform input, so its mutual information is also its capacity.

In [8]:
P = np.array([[0.4, 0.1], [0.1, 0.4]])
H_X = entropy_bits(P.sum(axis=1))
H_XY = entropy_bits(P)
H_X_given_Y = H_XY - entropy_bits(P.sum(axis=0))
print(f"H(X)     = {H_X:.10f} bits")
print(f"H(X,Y)   = {H_XY:.10f} bits")
print(f"H(X|Y)   = {H_X_given_Y:.10f} bits = H_b(0.2) = {binary_entropy(np.array(0.2)):.10f}")
print(f"I(X;Y)   = {mutual_information_bits(P):.10f} bits")
assert abs(mutual_information_bits(P) - (1.0 - binary_entropy(np.array(0.2)))) < 1e-14

H(X)     = 1.0000000000 bits
H(X,Y)   = 1.7219280949 bits
H(X|Y)   = 0.7219280949 bits = H_b(0.2) = 0.7219280949
I(X;Y)   = 0.2780719051 bits


### Problem L1.2 — Deriving the union form

**Statement.** Prove $I(X; Y) = H(X) + H(Y) - H(X, Y)$ from Definition 3.1, and deduce
$I(X; Y) \le \min\left( H(X), H(Y) \right)$ with its equality condition.

**Intuition.** Split the single logarithm into three, and each piece is an entropy.

**Solution.**

*Step 1 — split the logarithm.*

$$
I(X; Y) = \sum_{x, y} p(x, y) \left[ \log \frac{1}{p(x)} + \log \frac{1}{p(y)} - \log \frac{1}{p(x, y)} \right].
$$

*Step 2 — identify each sum.* Marginalizing $y$ in the first term gives $H(X)$; marginalizing $x$ in
the second gives $H(Y)$; the third is $-H(X, Y)$ by definition.

*Step 3 — the bound.* From $I = H(X) - H(X \mid Y)$ and $H(X \mid Y) \ge 0$ we get $I \le H(X)$, and
symmetrically $I \le H(Y)$.

*Step 4 — equality.* $I = H(X)$ holds exactly when $H(X \mid Y) = 0$, that is when $X$ is a
deterministic function of $Y$. Problem L0.3 is the instance: $I = H(Y)$ because $Y = X^2$ is
determined by $X$.

$$
\boxed{I(X; Y) = H(X) + H(Y) - H(X, Y) \le \min\left( H(X), H(Y) \right)}
$$

**Key takeaway.** Mutual information is the overlap of two entropy areas, so it can never exceed the
smaller of them.

In [9]:
draw = rng.random((4, 3)) ** 2
P = draw / draw.sum()
lhs = float((P * np.log2(P / np.outer(P.sum(axis=1), P.sum(axis=0)))).sum())
rhs = entropy_bits(P.sum(axis=1)) + entropy_bits(P.sum(axis=0)) - entropy_bits(P)
print(f"definition  = {lhs:.15f} bits")
print(f"union form  = {rhs:.15f} bits")
print(f"difference  = {abs(lhs - rhs):.3e} = {abs(lhs - rhs) / EPS:.2f} machine epsilons")
assert abs(lhs - rhs) < 1e-14

definition  = 0.754330567962974 bits
union form  = 0.754330567962974 bits
difference  = 1.110e-16 = 0.50 machine epsilons


### Problem L1.3 — The chain rule on XOR

**Statement.** Let $Y = X_1 \oplus X_2$ with $X_1, X_2$ independent fair bits. Compute
$I(X_1; Y)$, $I(X_2; Y \mid X_1)$ and $I(X_1, X_2; Y)$, and verify Theorem 4.3.

**Intuition.** Each input alone leaves the parity a fair coin; together they fix it.

**Solution.**

*Step 1 — one feature.* XOR with an independent fair bit randomizes, so $Y$ is a fair bit both
before and after conditioning on $X_1$: $I(X_1; Y) = 1 - 1 = 0$, and symmetrically
$I(X_2; Y) = 0$.

*Step 2 — the conditional term.* Given $X_1$, the map $X_2 \mapsto Y$ is a bijection, so
$H(Y \mid X_1, X_2) = 0$ while $H(Y \mid X_1) = 1$, giving $I(X_2; Y \mid X_1) = 1$ bit.

*Step 3 — the pair.* $I(X_1, X_2; Y) = H(Y) - H(Y \mid X_1, X_2) = 1 - 0 = 1$ bit.

*Step 4 — the chain rule.* $0 + 1 = 1$, as Theorem 4.3 requires.

$$
\boxed{I(X_1; Y) = I(X_2; Y) = 0, \qquad I(X_1, X_2; Y) = 1 \text{ bit}}
$$

**Key takeaway.** Information is not additive across features. Any filter that ranks by the marginal
$I(X_j; Y)$ discards both XOR features as worthless — the canonical failure of univariate selection.

In [10]:
P_xor = np.zeros((2, 2, 2))
for a in (0, 1):
    for b in (0, 1):
        P_xor[a, b, a ^ b] = 0.25
I1 = mutual_information_bits(P_xor.sum(axis=1))
I2 = mutual_information_bits(P_xor.sum(axis=0))
I_pair = mutual_information_bits(P_xor.reshape(4, 2))
I_2_given_1 = conditional_mi_bits(np.transpose(P_xor, (1, 2, 0)))
print(f"I(X1;Y) = {I1:.10f}   I(X2;Y) = {I2:.10f}")
print(f"I(X2;Y|X1) = {I_2_given_1:.10f}   I(X1,X2;Y) = {I_pair:.10f}")
print(f"chain-rule residual = {abs(I_pair - (I1 + I_2_given_1)):.3e}")
assert abs(I_pair - (I1 + I_2_given_1)) < 1e-14

I(X1;Y) = 0.0000000000   I(X2;Y) = 0.0000000000
I(X2;Y|X1) = 1.0000000000   I(X1,X2;Y) = 1.0000000000
chain-rule residual = 0.000e+00


### Problem L1.4 — Capacity of the binary symmetric channel

**Statement.** A channel flips each input bit with probability $\epsilon$. Compute $I(X; Y)$ for an
input with $\Pr[X = 1] = \pi$, and maximize over $\pi$ to obtain the capacity.

**Intuition.** The noise term does not depend on the input, so maximizing $I$ means maximizing the
output entropy.

**Solution.**

*Step 1 — isolate the noise.* Given $X$, the output differs by an independent
$\mathrm{Ber}(\epsilon)$ flip, so $H(Y \mid X) = H_b(\epsilon)$ for every $\pi$, and
$I(X; Y) = H(Y) - H_b(\epsilon)$.

*Step 2 — the output law.* $\Pr[Y = 1] = \pi(1 - \epsilon) + (1 - \pi)\epsilon$, so
$H(Y) = H_b\left( \pi(1-\epsilon) + (1-\pi)\epsilon \right)$.

*Step 3 — maximize.* $H(Y)$ is maximal at $\Pr[Y = 1] = \tfrac{1}{2}$, attained by $\pi = \tfrac{1}{2}$
whenever $\epsilon \neq \tfrac{1}{2}$, and then $H(Y) = 1$ bit.

*Step 4 — sanity checks.* $\epsilon = 0$ gives $C = 1$; $\epsilon = \tfrac{1}{2}$ gives $C = 0$;
$\epsilon = 1$ gives $C = 1$ again, because deterministic inversion is a bijection and
Theorem 4.4 says bijections lose nothing. At $\epsilon = 0.2$, $C = 0.278072$ bits, matching
Problem L1.1.

$$
\boxed{I(X; Y) = H_b\left( \pi(1-\epsilon) + (1-\pi)\epsilon \right) - H_b(\epsilon), \qquad C = 1 - H_b(\epsilon)}
$$

**Key takeaway.** Writing $I = H(Y) - H(Y \mid X)$ isolates the input-dependent part; capacity
problems are almost always solved by maximizing an output entropy.

In [11]:
def bsc(e):
    return np.array([[1.0 - e, e], [e, 1.0 - e]])


def I_of_input(pi, channel):
    r = np.array([1.0 - pi, pi])
    return mutual_information_bits(r[:, None] * channel)


pis = np.linspace(0.0, 1.0, 20001)
for eps in (0.0, 0.2, 0.5, 1.0):
    curve = np.array([I_of_input(p, bsc(eps)) for p in pis])
    where = (f"at pi = {pis[int(np.argmax(curve))]:.4f}" if curve.max() > 1e-12
             else "at every pi (I is identically zero)")
    print(f"  eps = {eps:.1f}:  max I = {curve.max():.10f} {where}"
          f"   1 - H_b(eps) = {1.0 - binary_entropy(np.array(eps)):.10f}")
    assert abs(curve.max() - (1.0 - binary_entropy(np.array(eps)))) < 1e-9

  eps = 0.0:  max I = 1.0000000000 at pi = 0.5000   1 - H_b(eps) = 1.0000000000


  eps = 0.2:  max I = 0.2780719051 at pi = 0.5000   1 - H_b(eps) = 0.2780719051


  eps = 0.5:  max I = 0.0000000000 at every pi (I is identically zero)   1 - H_b(eps) = 0.0000000000


  eps = 1.0:  max I = 1.0000000000 at pi = 0.5000   1 - H_b(eps) = 1.0000000000


### Problem L1.5 — Capacity of the binary erasure channel

**Statement.** A channel passes its input bit unchanged with probability $1 - \epsilon$ and outputs
an erasure symbol $?$ with probability $\epsilon$, never flipping. Compute the capacity.

**Intuition.** When the output is not an erasure the receiver knows the input exactly, so a fraction
$1 - \epsilon$ of the uses are noiseless and the rest carry nothing.

**Solution.**

*Step 1 — the erasure indicator.* Let $E = \mathbf{1}\lbrace Y = \; ? \rbrace$. It is independent of
$X$ by construction, so $H(X \mid Y) = H(X \mid Y, E)$ and

$$
H(X \mid Y) = \Pr[E = 0]\, H(X \mid Y, E = 0) + \Pr[E = 1]\, H(X \mid Y, E = 1).
$$

*Step 2 — evaluate the two branches.* On $E = 0$ the output reveals $X$, so the first conditional
entropy is $0$. On $E = 1$ the output is uninformative, so the second is $H(X)$.

*Step 3 — assemble.* $H(X \mid Y) = \epsilon H(X)$, hence

$$
I(X; Y) = H(X) - \epsilon H(X) = (1 - \epsilon) H(X).
$$

*Step 4 — maximize.* $H(X) \le 1$ bit with equality at the uniform input, so
$C = 1 - \epsilon$ bits per use.

$$
\boxed{C_{\mathrm{BEC}} = 1 - \epsilon \text{ bits per use, attained by the uniform input}}
$$

**Key takeaway.** Erasures are much gentler than flips: at $\epsilon = 0.2$ the erasure channel
carries $0.8$ bits while the symmetric channel of Problem L1.4 carries only $0.278072$, because an
erasure announces itself and a flip does not.

In [12]:
for eps in (0.1, 0.2, 0.3, 0.5):
    W_bec = np.array([[1.0 - eps, eps, 0.0], [0.0, eps, 1.0 - eps]])
    curve = np.array([I_of_input(p, W_bec) for p in pis])
    print(f"  eps = {eps:.1f}:  max I = {curve.max():.10f} bits at pi = {pis[int(np.argmax(curve))]:.4f}"
          f"   1 - eps = {1.0 - eps:.10f}   BSC would give {1.0 - binary_entropy(np.array(eps)):.6f}")
    assert abs(curve.max() - (1.0 - eps)) < 1e-9

  eps = 0.1:  max I = 0.9000000000 bits at pi = 0.5000   1 - eps = 0.9000000000   BSC would give 0.531004


  eps = 0.2:  max I = 0.8000000000 bits at pi = 0.5000   1 - eps = 0.8000000000   BSC would give 0.278072


  eps = 0.3:  max I = 0.7000000000 bits at pi = 0.5000   1 - eps = 0.7000000000   BSC would give 0.118709


  eps = 0.5:  max I = 0.5000000000 bits at pi = 0.5000   1 - eps = 0.5000000000   BSC would give 0.000000


### Problem L1.6 — Gaussian mutual information and SNR

**Statement.** For jointly Gaussian $(X, Y)$ with correlation $\rho$, compute $I(X; Y)$ in bits at
$\rho = 0.5, 0.9, 0.99$, and convert each to the equivalent signal-to-noise ratio.

**Intuition.** The information is a logarithm of $1/(1 - \rho^2)$, so it explodes only as $\rho$
approaches one.

**Solution.**

*Step 1 — the formula.* Theorem 4.6 gives $I(X; Y) = -\tfrac{1}{2}\log_2(1 - \rho^2)$ bits.

*Step 2 — evaluate.*

| $\rho$ | $1 - \rho^2$ | $I(X; Y)$ (bits) |
|---|---|---|
| $0.50$ | $0.7500$ | $0.207519$ |
| $0.90$ | $0.1900$ | $1.197964$ |
| $0.99$ | $0.0199$ | $2.825544$ |

*Step 3 — match against the channel formula.* Setting
$\tfrac{1}{2}\log_2(1 + \mathrm{SNR}) = -\tfrac{1}{2}\log_2(1 - \rho^2)$ gives
$1 + \mathrm{SNR} = 1/(1 - \rho^2)$, that is

$$
\mathrm{SNR} = \frac{\rho^{2}}{1 - \rho^{2}} .
$$

*Step 4 — the three values.* $0.333333$ ($-4.77$ dB), $4.263158$ ($6.30$ dB) and
$49.251256$ ($16.92$ dB).

$$
\boxed{I = -\tfrac{1}{2}\log_2\left(1 - \rho^2\right) \text{ bits}, \qquad \mathrm{SNR} = \frac{\rho^{2}}{1 - \rho^{2}}}
$$

**Key takeaway.** Correlation buys information slowly and then explosively: $\rho = 0.9 \to 0.99$
costs a tenfold SNR increase for $1.63$ extra bits.

In [13]:
for rho in (0.5, 0.9, 0.99):
    I_bits = -0.5 * np.log2(1.0 - rho ** 2)
    snr = rho ** 2 / (1.0 - rho ** 2)
    print(f"  rho = {rho:.2f}:  1 - rho^2 = {1 - rho ** 2:.4f}   I = {I_bits:.6f} bits"
          f"   SNR = {snr:.6f} = {10 * np.log10(snr):.2f} dB")
    assert abs(0.5 * np.log2(1.0 + snr) - I_bits) < 1e-12
print(f"  extra bits from rho = 0.9 to rho = 0.99: "
      f"{-0.5 * np.log2(1 - 0.99 ** 2) + 0.5 * np.log2(1 - 0.9 ** 2):.6f}")

  rho = 0.50:  1 - rho^2 = 0.7500   I = 0.207519 bits   SNR = 0.333333 = -4.77 dB
  rho = 0.90:  1 - rho^2 = 0.1900   I = 1.197964 bits   SNR = 4.263158 = 6.30 dB
  rho = 0.99:  1 - rho^2 = 0.0199   I = 2.825544 bits   SNR = 49.251256 = 16.92 dB
  extra bits from rho = 0.9 to rho = 0.99: 1.627580


### Problem L1.7 — Conditioning can increase mutual information

**Statement.** Let $X, Y$ be independent fair bits and $Z = X \oplus Y$. Compute $I(X; Y)$ and
$I(X; Y \mid Z)$, and reconcile the answer with "conditioning reduces entropy".

**Intuition.** $Z$ is a common effect of $X$ and $Y$; observing an effect couples its causes.

**Solution.**

*Step 1 — unconditional.* $X$ and $Y$ are independent by construction, so $I(X; Y) = 0$.

*Step 2 — conditional.* Given $Z = z$ the pair satisfies $Y = X \oplus z$, so $Y$ is a deterministic
function of $X$ and $I(X; Y \mid Z) = H(X \mid Z) - H(X \mid Y, Z) = 1 - 0 = 1$ bit.

*Step 3 — reconcile.* The monotone statement of Theorem 4.2 is $H(X \mid Y) \le H(X)$, about
*entropy*. It does not imply $I(X; Y \mid Z) \le I(X; Y)$, because the conditional mutual information
is a difference of two entropies that both shrink and the second can shrink more.

*Step 4 — the Venn consequence.* The interaction information
$I(X; Y) - I(X; Y \mid Z) = -1$ bit is negative, so no three-circle area diagram represents this
triple.

$$
\boxed{I(X; Y) = 0 \;\lt\; I(X; Y \mid Z) = 1 \text{ bit}}
$$

**Key takeaway.** Conditional independence is not monotone in the conditioning set. Stratifying on a
downstream variable manufactures dependence — Berkson's paradox, and the reason "explaining away" is
a real statistical effect rather than a metaphor.

In [14]:
I_uncond = mutual_information_bits(P_xor.sum(axis=2))
I_cond = conditional_mi_bits(P_xor)
print(f"I(X;Y)     = {I_uncond:.10f} bits")
print(f"I(X;Y|Z)   = {I_cond:.10f} bits")
print(f"interaction information = {I_uncond - I_cond:+.10f} bits")
assert I_cond > I_uncond

I(X;Y)     = 0.0000000000 bits
I(X;Y|Z)   = 1.0000000000 bits
interaction information = -1.0000000000 bits


### Problem L1.8 — Invariance under invertible maps

**Statement.** Prove $I\left( f(X); g(Y) \right) = I(X; Y)$ for invertible $f$ and $g$, and
$I\left( f(X); Y \right) \le I(X; Y)$ for arbitrary $f$.

**Intuition.** A relabelling is a lossless recoding, and the data-processing inequality applies in
both directions when it can be undone.

**Solution.**

*Step 1 — arbitrary $f$.* The chain $Y \to X \to f(X)$ holds because $f(X)$ depends on $Y$ only
through $X$, so Theorem 4.4 gives $I\left( f(X); Y \right) \le I(X; Y)$.

*Step 2 — invertible $f$.* Then $X = f^{-1}(f(X))$ is a function of $f(X)$, so the reverse chain
$Y \to f(X) \to X$ also holds and $I(X; Y) \le I\left( f(X); Y \right)$. The two inequalities
together give equality.

*Step 3 — the second argument.* Repeat Steps 1 and 2 with the roles of $X$ and $Y$ exchanged.

*Step 4 — the continuous version.* For densities, the change of variables inserts the Jacobian
factor $\lvert \det J_f \rvert$ into $h(f(X))$ and into $h\left( f(X) \mid Y \right)$ identically,
so it cancels in the difference — the analytic form of the same fact.

$$
\boxed{I\left( f(X); g(Y) \right) = I(X; Y) \text{ for invertible } f, g; \qquad I\left( f(X); Y \right) \le I(X; Y) \text{ in general}}
$$

**Key takeaway.** Mutual information ignores units, monotone rescalings and lossless re-encodings,
unlike correlation, which changes under any nonlinear reparameterization.

In [15]:
P = np.array([[0.30, 0.10, 0.05], [0.05, 0.25, 0.05], [0.05, 0.05, 0.10]])
I_base = mutual_information_bits(P)
perm = np.array([2, 0, 1])
I_perm = mutual_information_bits(P[perm][:, perm])
merged = np.column_stack([P[:, 0] + P[:, 1], P[:, 2]])
I_merged = mutual_information_bits(merged)
print(f"I(X;Y)                          = {I_base:.12f} bits")
print(f"after relabelling both alphabets = {I_perm:.12f} bits   difference {abs(I_base - I_perm):.3e}")
print(f"after merging two values of Y    = {I_merged:.12f} bits   lost {I_base - I_merged:.6f} bits")
assert abs(I_base - I_perm) < 1e-14
assert I_merged < I_base

I(X;Y)                          = 0.268858395456 bits
after relabelling both alphabets = 0.268858395456 bits   difference 4.441e-16
after merging two values of Y    = 0.088376371735 bits   lost 0.180482 bits


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Information gain of a decision-tree split

**Statement.** A node holds $100$ examples, $60$ positive and $40$ negative. A candidate split on
feature $A$ produces a left child with $50$ examples ($45$ positive, $5$ negative) and a right child
with $50$ ($15$ positive, $35$ negative). Compute the information gain and the gain ratio.

**Intuition.** Information gain is the label entropy before the split minus the average label
entropy after it, which is exactly $I(Y; A)$.

**Solution.**

*Step 1 — parent entropy.* $H(Y) = H_b(0.6) = 0.970951$ bits.

*Step 2 — children.* Left: $H_b(0.9) = 0.468996$. Right: $H_b(0.3) = 0.881291$.

*Step 3 — conditional entropy.* Both children hold half the data, so

$$
H(Y \mid A) = 0.5(0.468996) + 0.5(0.881291) = 0.675143 \text{ bits}.
$$

*Step 4 — gain.* $\mathrm{IG} = I(Y; A) = 0.970951 - 0.675143 = 0.295807$ bits.

*Step 5 — gain ratio.* The split itself has entropy $H(A) = H_b(0.5) = 1$ bit, so the gain ratio is
$0.295807 / 1 = 0.295807$. The correction bites when a candidate has many values: an identifier-like
feature reaches $\mathrm{IG} = H(Y)$ but also $H(A) = \log_2 100$, so its ratio collapses.

$$
\boxed{\mathrm{IG} = 0.295807 \text{ bits}, \qquad \mathrm{GainRatio} = 0.295807}
$$

**Key takeaway.** The information gain of ID3 and C4.5 is literally $I(Y; A)$, and the gain ratio is
the standard patch for the bound $I(Y; A) \le H(A)$ favouring high-cardinality splits.

In [16]:
P_split = np.array([[0.45, 0.05], [0.15, 0.35]])
H_Y = entropy_bits(P_split.sum(axis=0))
IG = mutual_information_bits(P_split.T)
H_A = entropy_bits(P_split.sum(axis=1))
print(f"H(Y)      = H_b(0.6) = {H_Y:.6f} bits")
print(f"H_b(0.9)  = {binary_entropy(np.array(0.9)):.6f},  H_b(0.3) = {binary_entropy(np.array(0.3)):.6f}")
print(f"H(Y | A)  = {H_Y - IG:.6f} bits")
print(f"IG        = {IG:.6f} bits,   gain ratio = {IG / H_A:.6f}")
assert abs(IG - 0.2958073480446819) < 1e-12

H(Y)      = H_b(0.6) = 0.970951 bits
H_b(0.9)  = 0.468996,  H_b(0.3) = 0.881291
H(Y | A)  = 0.675143 bits
IG        = 0.295807 bits,   gain ratio = 0.295807


### Problem L2.2 — Plug-in bias and a permutation null

**Statement.** Two *independent* variables are each discretized into $10$ bins and $N = 200$ paired
samples are collected. Predict the plug-in estimate of $I(X; Y)$, and design a test that is not
fooled by it.

**Intuition.** The estimator is nonnegative by construction, so its sampling noise cannot cancel and
has to show up as positive information.

**Solution.**

*Step 1 — the bias.* Proposition 4.11 with $K = L = 10$ and $N = 200$ predicts

$$
\mathbb{E}\left[ \hat{I} \right] \approx 0 + \frac{9 \times 9}{2 \times 200} = 0.2025 \text{ nats}.
$$

*Step 2 — in bits.* $0.2025 / \ln 2 = 0.292146$ bits of pure artefact, larger than the real signal
in Problem L2.1.

*Step 3 — why.* Two hundred samples cannot fill a hundred-cell table evenly; the random imbalance
looks like dependence, and $\hat{I} \ge 0$ means the fluctuation only ever adds.

*Step 4 — a defensible test.*

1. Compute $\hat{I}_{\text{obs}}$ on the true pairing.
2. Permute $Y$ against $X$ $B$ times and recompute. This destroys dependence while preserving both
   marginals and the table shape, so the null carries exactly the same bias.
3. Report the p-value $\frac{1 + \#\lbrace b : \hat{I}^{(b)} \ge \hat{I}_{\text{obs}} \rbrace}{1 + B}$
   and the bias-corrected effect size $\hat{I}_{\text{obs}} - \overline{\hat{I}^{(b)}}$.

$$
\boxed{\text{spurious } \hat{I} \approx 0.2025 \text{ nats} = 0.292146 \text{ bits}; \text{ calibrate against a permutation null}}
$$

**Key takeaway.** A positive estimated mutual information means nothing on its own. The estimator's
floor sits well above zero whenever the table is large relative to the sample, and the permutation
null is the only baseline that reproduces that floor exactly.

In [17]:
def plugin_mi_nats(xs, ys, kx, ky):
    counts = np.zeros((kx, ky))
    np.add.at(counts, (xs, ys), 1.0)
    Q = counts / counts.sum()
    qx, qy = Q.sum(axis=1), Q.sum(axis=0)
    nz = Q > 0.0
    return float((Q[nz] * np.log(Q[nz] / np.outer(qx, qy)[nz])).sum())


kx = ky = 10
N, B = 200, 1000
print(f"predicted bias (K-1)(L-1)/(2N) = {(kx - 1) * (ky - 1) / (2 * N):.6f} nats"
      f" = {(kx - 1) * (ky - 1) / (2 * N) / LOG2:.6f} bits")

xs = rng.integers(0, kx, N)
ys = rng.integers(0, ky, N)
obs = plugin_mi_nats(xs, ys, kx, ky)
null = np.array([plugin_mi_nats(xs, rng.permutation(ys), kx, ky) for _ in range(B)])
p_val = (1 + np.sum(null >= obs)) / (1 + B)
print(f"independent pairs: I_hat = {obs:.4f} nats, null mean = {null.mean():.4f},"
      f" p = {p_val:.4f}, corrected = {obs - null.mean():+.4f}")

xs2 = rng.integers(0, kx, N)
ys2 = np.where(rng.random(N) < 0.3, rng.integers(0, ky, N), xs2)
obs2 = plugin_mi_nats(xs2, ys2, kx, ky)
null2 = np.array([plugin_mi_nats(xs2, rng.permutation(ys2), kx, ky) for _ in range(B)])
p_val2 = (1 + np.sum(null2 >= obs2)) / (1 + B)
print(f"70% dependent    : I_hat = {obs2:.4f} nats, null mean = {null2.mean():.4f},"
      f" p = {p_val2:.4f}, corrected = {obs2 - null2.mean():+.4f}")
assert p_val > 0.05 and p_val2 < 0.05

predicted bias (K-1)(L-1)/(2N) = 0.202500 nats = 0.292146 bits


independent pairs: I_hat = 0.1944 nats, null mean = 0.2364, p = 0.8791, corrected = -0.0420


70% dependent    : I_hat = 1.3098 nats, null mean = 0.2348, p = 0.0010, corrected = +1.0751


### Problem L2.3 — Batch size and the InfoNCE ceiling

**Statement.** A contrastive model trains with batch size $K$ and reports a converged loss of
$\mathcal{L}_{\mathrm{NCE}} = 0.5$ nats. Compute the certified lower bound on $I(X; Y)$ for
$K = 256$ and $K = 65536$, and state what may and may not be concluded.

**Intuition.** The certificate is chance level minus achieved loss, and chance level is $\log K$.

**Solution.**

*Step 1 — apply Theorem 4.8.* $I(X; Y) \ge \log K - \mathcal{L}_{\mathrm{NCE}}$ with the
unnormalized loss of Definition 3.6.

For $K = 256$: $\ln 256 = 5.545177$, so $I \ge 5.045177$ nats $= 7.278652$ bits.

For $K = 65536$: $\ln 65536 = 11.090355$, so $I \ge 10.590355$ nats $= 15.278652$ bits.

*Step 2 — the ceiling.* Since $\mathcal{L}_{\mathrm{NCE}} \ge 0$, the bound can never exceed
$\log K$: $5.545177$ and $11.090355$ nats respectively. Doubling the batch buys exactly
$\ln 2 = 0.693147$ nats of certifiable information — a logarithmic return.

*Step 3 — what cannot be concluded.* The true $I(X; Y)$ may be far larger; a small loss at large $K$
only says the critic ranks the positive above $K-1$ random negatives. "Our representation has
$15$ bits of mutual information" is unjustified; "at least $15.28$ bits, as certified by this
estimator" is correct. Conversely a loss near $\log K$ does prove the representation is nearly
uninformative *for this critic family* — Section 7.5 of the theory notebook shows the constant
critic returning a certificate of zero.

$$
\boxed{I \ge \log K - \mathcal{L}_{\mathrm{NCE}}: \; 5.045177 \text{ nats at } K = 256, \; 10.590355 \text{ nats at } K = 65536}
$$

**Key takeaway.** Every large-batch, memory-bank and momentum-encoder trick in contrastive learning
is a fight against the $\log K$ ceiling. The mathematics predicted the engineering.

In [18]:
loss = 0.5
for K in (256, 65536):
    bound = np.log(K) - loss
    print(f"  K = {K:6d}:  log K = {np.log(K):.6f} nats,  bound = {bound:.6f} nats"
          f" = {bound / LOG2:.6f} bits,  ceiling = {np.log(K) / LOG2:.4f} bits")
print(f"  doubling the batch adds ln 2 = {np.log(2.0):.6f} nats to the ceiling")
assert abs((np.log(256) - loss) - 5.045177444479562) < 1e-12

  K =    256:  log K = 5.545177 nats,  bound = 5.045177 nats = 7.278652 bits,  ceiling = 8.0000 bits
  K =  65536:  log K = 11.090355 nats,  bound = 10.590355 nats = 15.278652 bits,  ceiling = 16.0000 bits
  doubling the batch adds ln 2 = 0.693147 nats to the ceiling


### Problem L2.4 — Relevance, redundancy, and mRMR

**Statement.** Let $Y \sim \mathrm{Ber}(1/2)$ and build three features by independent noisy copies:
$X_1 = Y \oplus N_1$ with $N_1 \sim \mathrm{Ber}(0.11)$; $X_2 = X_1 \oplus N_2$ with
$N_2 \sim \mathrm{Ber}(0.05)$; and $X_3 = Y \oplus N_3$ with $N_3 \sim \mathrm{Ber}(0.25)$. Select
two features by relevance ranking and by mRMR, and compare the joint information each pair carries.

**Intuition.** $X_2$ is a degraded copy of $X_1$, so by the data-processing inequality it can add
nothing at all once $X_1$ is in hand.

**Solution.**

*Step 1 — relevances.* Each feature is $Y$ through a binary symmetric channel, so
$I(X_j; Y) = 1 - H_b(\epsilon_j)$ with the effective crossovers $0.11$, $0.11(0.95) + 0.89(0.05) = 0.149$
and $0.25$:

$$
I(X_1; Y) = 0.500084, \qquad I(X_2; Y) = 0.392668, \qquad I(X_3; Y) = 0.188722 \ \text{bits}.
$$

*Step 2 — redundancies.* $X_1$ and $X_2$ differ by $N_2$ alone, so $I(X_1; X_2) = 1 - H_b(0.05) = 0.713603$
bits. $X_1$ and $X_3$ see $Y$ through independent noise, giving
$I(X_1; X_3) = 1 - H_b(0.305) = 0.112683$ bits.

*Step 3 — relevance ranking.* Sorting by relevance selects $\lbrace X_1, X_2 \rbrace$.

*Step 4 — mRMR.* Starting from $S = \lbrace X_1 \rbrace$, the greedy score is

$$
J(X_j) = I(X_j; Y) - \frac{1}{\lvert S \rvert}\sum_{k \in S} I(X_j; X_k),
$$

so $J(X_2) = 0.392668 - 0.713603 = -0.320935$ and $J(X_3) = 0.188722 - 0.112683 = 0.076039$. mRMR
selects $\lbrace X_1, X_3 \rbrace$.

*Step 5 — the joint information of each pair.* Because $Y \to X_1 \to X_2$ is a Markov chain,
Theorem 4.4 gives $I(X_2; Y \mid X_1) = 0$ exactly, so

$$
I(X_1, X_2; Y) = I(X_1; Y) = 0.500084 \ \text{bits}, \qquad I(X_1, X_3; Y) = 0.576123 \ \text{bits}.
$$

The second pair adds $I(X_3; Y \mid X_1) = 0.076039$ bits, which the mRMR surrogate happened to
reproduce to six decimals here.

$$
\boxed{\text{relevance ranking: } 0.500084 \text{ bits}; \qquad \text{mRMR: } 0.576123 \text{ bits}}
$$

**Key takeaway.** Feature *sets* are scored by the chain rule $I(X_1; Y) + I(X_3; Y \mid X_1)$ of
Theorem 4.3, not by summing marginal relevances. Redundancy penalties are a cheap surrogate for the
conditional term, and the second-best feature can contribute exactly nothing.

In [19]:
p1, p2, p3 = 0.11, 0.05, 0.25
P4 = np.zeros((2, 2, 2, 2))          # axes: Y, X1, X2, X3
for y in (0, 1):
    for n1 in (0, 1):
        for n2 in (0, 1):
            for n3 in (0, 1):
                w = 0.5 * (p1 if n1 else 1 - p1) * (p2 if n2 else 1 - p2) * (p3 if n3 else 1 - p3)
                P4[y, y ^ n1, (y ^ n1) ^ n2, y ^ n3] += w
assert abs(P4.sum() - 1.0) < 1e-14


def pair(axis_a, axis_b):
    drop = tuple(a for a in range(4) if a not in (axis_a, axis_b))
    Q = P4.sum(axis=drop)
    return Q if axis_a < axis_b else Q.T


rel = [mutual_information_bits(pair(j, 0)) for j in (1, 2, 3)]
print(f"relevances  I(X1;Y) = {rel[0]:.6f}   I(X2;Y) = {rel[1]:.6f}   I(X3;Y) = {rel[2]:.6f} bits")
red12 = mutual_information_bits(pair(1, 2))
red13 = mutual_information_bits(pair(1, 3))
print(f"redundancy  I(X1;X2) = {red12:.6f}   I(X1;X3) = {red13:.6f} bits")
print(f"mRMR scores J(X2) = {rel[1] - red12:+.6f}   J(X3) = {rel[2] - red13:+.6f}")
joint12 = mutual_information_bits(P4.sum(axis=3).transpose(1, 2, 0).reshape(4, 2))
joint13 = mutual_information_bits(P4.sum(axis=2).transpose(1, 2, 0).reshape(4, 2))
print(f"I(X1,X2;Y) = {joint12:.6f} bits   I(X1,X3;Y) = {joint13:.6f} bits")
print(f"I(X2;Y|X1) = {joint12 - rel[0]:.3e}   I(X3;Y|X1) = {joint13 - rel[0]:.6f} bits")
assert abs(joint12 - rel[0]) < 1e-14
assert joint13 > joint12

relevances  I(X1;Y) = 0.500084   I(X2;Y) = 0.392668   I(X3;Y) = 0.188722 bits
redundancy  I(X1;X2) = 0.713603   I(X1;X3) = 0.112683 bits
mRMR scores J(X2) = -0.320935   J(X3) = +0.076039
I(X1,X2;Y) = 0.500084 bits   I(X1,X3;Y) = 0.576123 bits
I(X2;Y|X1) = 0.000e+00   I(X3;Y|X1) = 0.076039 bits


### Problem L2.5 — The data-processing inequality as a fairness certificate

**Statement.** A pipeline is $S \to X \to Z \to \hat{Y}$, where $S$ is a protected attribute, $Z$ a
learned representation and $\hat{Y}$ any downstream head. Show that $I(S; Z) = 0$ forces
$I(S; \hat{Y}) = 0$, and say what an audited $I(S; Z) = 0.05$ bits permits, for binary $S$ with
$H(S) = 1$ bit.

**Intuition.** The head sees only $Z$, so it cannot know more about $S$ than $Z$ does.

**Solution.**

*Step 1 — apply Theorem 4.4.* $S \to Z \to \hat{Y}$ is a Markov chain, so
$I(S; \hat{Y}) \le I(S; Z)$.

*Step 2 — the certificate.* If $I(S; Z) = 0$ then $0 \le I(S; \hat{Y}) \le 0$, so the prediction is
independent of $S$ for **every** head, including ones trained later by someone else. That
architecture-independence is what makes representation-level constraints attractive.

*Step 3 — a nonzero budget.* With $I(S; Z) = 0.05$ bits, $H(S \mid Z) = 0.95$ bits. Corollary 4.7a
with $K = 2$ gives $P_e \ge (1 - 0.05 - 1)/1 \lt 0$, which is vacuous, so use Theorem 4.7 directly:
for binary $S$ the term $P_e \log(K-1)$ vanishes and Fano reduces to $H_b(P_e) \ge H(S \mid Z)$.
Solving $H_b(P_e) = 0.95$ gives the two roots $0.369128$ and $0.630872$, and the relevant one is the
smaller:

$$
P_e \ge 0.369128, \qquad \text{adversary accuracy} \le 0.630872 .
$$

*Step 4 — caveats.* Independence at the representation level is demographic parity, not equalized
odds; forcing $I(S; Z) = 0$ destroys legitimate accuracy when $S$ genuinely predicts $Y$; and any
estimate of $I(S; Z)$ inherits every bias of Problem L2.2.

$$
\boxed{I(S; \hat{Y}) \le I(S; Z); \quad I(S; Z) = 0.05 \text{ bits} \implies \text{adversary accuracy} \le 63.09\%}
$$

**Key takeaway.** The data-processing inequality turns an information budget into a hard,
downstream-proof leakage bound, and Fano converts the bits into an attacker's accuracy ceiling.

In [20]:
from scipy.optimize import brentq

for budget in (0.0, 0.05, 0.25, 1.0):
    H_S_given_Z = 1.0 - budget
    pe = brentq(lambda p: binary_entropy(np.array(p)) - H_S_given_Z, 0.0, 0.5, xtol=1e-14)
    print(f"  I(S;Z) = {budget:.2f} bits -> H(S|Z) = {H_S_given_Z:.2f}"
          f" -> Pe >= {pe:.6f}, accuracy <= {1.0 - pe:.6f}")
pe_005 = brentq(lambda p: binary_entropy(np.array(p)) - 0.95, 0.0, 0.5, xtol=1e-14)
print(f"  H_b(0.373) = {binary_entropy(np.array(0.373)):.6f}, not 0.95 - the root is"
      f" {pe_005:.6f}")
assert abs(pe_005 - 0.36912774898451184) < 1e-9

  I(S;Z) = 0.00 bits -> H(S|Z) = 1.00 -> Pe >= 0.500000, accuracy <= 0.500000
  I(S;Z) = 0.05 bits -> H(S|Z) = 0.95 -> Pe >= 0.369128, accuracy <= 0.630872
  I(S;Z) = 0.25 bits -> H(S|Z) = 0.75 -> Pe >= 0.214502, accuracy <= 0.785498
  I(S;Z) = 1.00 bits -> H(S|Z) = 0.00 -> Pe >= 0.000000, accuracy <= 1.000000
  H_b(0.373) = 0.952948, not 0.95 - the root is 0.369128


### Problem L2.6 — MINE in practice: why the gradient needs debiasing

**Statement.** The MINE estimator maximizes
$\hat{I}_\theta = \mathbb{E}_{p(x,y)}[f_\theta] - \log \widehat{\mathbb{E}}_{p(x)p(y)}\left[ e^{f_\theta} \right]$
over minibatches. Show that the minibatch objective is biased, that the naive gradient is biased,
and give the standard fix.

**Intuition.** A logarithm of an unbiased average is not an unbiased logarithm, and the gap is a
Jensen gap.

**Solution.**

*Step 1 — bias of the objective.* Let $\hat{m} = \frac{1}{B}\sum_b e^{f(x_b, y_b')}$, an unbiased
estimate of $m = \mathbb{E}_{p(x)p(y)}\left[ e^{f} \right]$. Concavity of the logarithm gives

$$
\mathbb{E}\left[ \log \hat{m} \right] \le \log \mathbb{E}\left[ \hat{m} \right] = \log m,
$$

so $-\log \hat{m}$ is biased *upward* and $\hat{I}_\theta$ overestimates the true
Donsker-Varadhan value on any finite batch. A delta-method expansion puts the gap at
$\operatorname{Var}(e^f) / (2 B m^2)$, so it decays like $1/B$.

*Step 2 — bias of the gradient.* Differentiating the second term,

$$
\nabla_\theta \log \hat{m} = \frac{\widehat{\mathbb{E}}\left[ e^{f_\theta} \nabla_\theta f_\theta \right]}{\widehat{\mathbb{E}}\left[ e^{f_\theta} \right]} .
$$

Numerator and denominator come from the *same* minibatch, so the ratio is a biased estimate of
$\mathbb{E}\left[ e^{f}\nabla f \right] / \mathbb{E}\left[ e^{f} \right]$: the correlation between
them does not vanish, and the variance of $e^{f}$ grows exponentially in $f$.

*Step 3 — the fix.* Replace the denominator by an exponential moving average
$m_t = (1 - \alpha) m_{t-1} + \alpha \widehat{\mathbb{E}}\left[ e^{f_\theta} \right]$ maintained
across steps and **not** differentiated:

$$
\nabla_\theta \hat{I} \approx \widehat{\mathbb{E}}_{p(x,y)}\left[ \nabla_\theta f_\theta \right] - \frac{\widehat{\mathbb{E}}\left[ e^{f_\theta}\nabla_\theta f_\theta \right]}{m_t} .
$$

This decouples numerator from denominator and restores an asymptotically unbiased gradient.
Complementary tricks: clip $f_\theta$; use the NWJ objective
$\mathbb{E}_{p(x,y)}[f] - e^{-1}\mathbb{E}_{p(x)p(y)}\left[ e^{f} \right]$, which is unbiased by
construction; or fall back on InfoNCE when stability matters more than tightness.

$$
\boxed{\log\text{-of-mean-exp is Jensen-biased with a gap of order } 1/B; \text{ use an EMA denominator or the NWJ bound}}
$$

**Key takeaway.** Variational estimates fail in two opposite ways at once — high variance from the
exponential and optimistic bias from the logarithm — so an unaudited MINE curve is not evidence of
anything.

In [21]:
sigma = 1.0
log_m_true = sigma ** 2 / 2.0        # e^f lognormal: log E[e^f] = sigma^2 / 2
batches = np.array([4, 8, 16, 32, 64])
trials = 100_000
gaps = np.empty(len(batches))
for i, B_ in enumerate(batches):
    m_hat = np.exp(sigma * rng.standard_normal((trials, int(B_)))).mean(axis=1)
    gaps[i] = log_m_true - np.log(m_hat).mean()
predicted = (np.e - 1.0) / (2.0 * batches)   # Var(e^f)/(2 B m^2) for the lognormal
order = np.polyfit(np.log(batches), np.log(gaps), 1)[0]
print("   B     measured gap   delta-method   ratio")
for B_, g, p in zip(batches, gaps, predicted):
    print(f"  {B_:3d}     {g:.6f}       {p:.6f}     {g / p:.4f}")
print(f"  observed order in B = {order:.4f}   predicted -1")
assert np.all(gaps > 0.0)
assert abs(order + 1.0) < 0.2

   B     measured gap   delta-method   ratio
    4     0.161411       0.214785     0.7515
    8     0.091023       0.107393     0.8476
   16     0.049128       0.053696     0.9149
   32     0.025236       0.026848     0.9400
   64     0.013522       0.013424     1.0073
  observed order in B = -0.9005   predicted -1


### Problem L2.7 — Physics: capacity of a thermal-noise-limited link

**Statement.** A receiver has bandwidth $B = 10$ MHz and system noise temperature $T = 290$ K, and
the received signal power is $P = 1$ pW. Using the Boltzmann constant
$k_B = 1.380649 \times 10^{-23}$ J/K, compute the noise power, the signal-to-noise ratio in decibels
and the Shannon-Hartley capacity.

**Intuition.** Thermal agitation of the electrons in the front end sets an unavoidable noise floor
$k_B T$ per hertz, and Theorem 4.6 converts that floor into a bit rate.

**Solution.**

*Step 1 — the noise floor.* The available noise power of a resistor at temperature $T$ into a
bandwidth $B$ is $N = k_B T B$, so

$$
N = 1.380649 \times 10^{-23} \times 290 \times 10^{7} = 4.0039 \times 10^{-14} \ \text{W}.
$$

*Step 2 — the signal-to-noise ratio.*

$$
\mathrm{SNR} = \frac{P}{N} = \frac{10^{-12}}{4.0039 \times 10^{-14}} = 24.976 = 13.98 \ \text{dB}.
$$

*Step 3 — apply Theorem 4.6 per degree of freedom.* A bandlimited channel supplies $2B$ real
dimensions per second, each an additive white Gaussian noise use of capacity
$\tfrac{1}{2}\log_2(1 + \mathrm{SNR})$ bits, so the rate is

$$
C = B \log_2\left( 1 + \mathrm{SNR} \right) = 10^{7} \times \log_2(25.976) = 46.991 \ \text{Mbit/s}.
$$

*Step 4 — read the scaling.* Doubling the transmitter power adds only
$B \log_2\left( \frac{1 + 2\,\mathrm{SNR}}{1 + \mathrm{SNR}} \right) = 9.720$ Mbit/s here, while
doubling the bandwidth nearly doubles the rate — except that $N = k_B T B$ grows with $B$ too, which
is why wideband links are noise-limited rather than power-limited.

$$
\boxed{N = 4.0039 \times 10^{-14} \ \text{W}, \quad \mathrm{SNR} = 13.98 \ \text{dB}, \quad C = 46.991 \ \text{Mbit/s}}
$$

**Key takeaway.** Capacity here is set by two physical constants and a temperature; no modulation or
coding scheme beats it, by Theorem 4.10.

In [22]:
k_B, T_sys, B_hz, P_rx = 1.380649e-23, 290.0, 10.0e6, 1.0e-12
N_pow = k_B * T_sys * B_hz
snr = P_rx / N_pow
C_link = B_hz * np.log2(1.0 + snr)
print(f"  N = kB T B         = {N_pow:.4e} W")
print(f"  SNR                = {snr:.3f} = {10 * np.log10(snr):.2f} dB")
print(f"  C = B log2(1+SNR)  = {C_link / 1e6:.3f} Mbit/s")
print(f"  doubling the power adds {(B_hz * np.log2(1 + 2 * snr) - C_link) / 1e6:.3f} Mbit/s")
assert abs(C_link / 1e6 - 46.99094077296952) < 1e-9

  N = kB T B         = 4.0039e-14 W
  SNR                = 24.976 = 13.98 dB
  C = B log2(1+SNR)  = 46.991 Mbit/s
  doubling the power adds 9.720 Mbit/s


### Problem L2.8 — Physics: mutual information between two spins in the Ising chain

**Statement.** In the one-dimensional Ising chain with nearest-neighbour coupling $J$ and no
external field, the spin-spin correlation at separation $r$ is
$\langle \sigma_i \sigma_{i+r} \rangle = t^{\,r}$ with $t = \tanh\left( J / k_B T \right)$, and each
spin is $\pm 1$ with probability $\tfrac{1}{2}$. Compute $I(\sigma_1; \sigma_4)$ at
$J / k_B T = 1$, and compare the decay length of the information with the correlation length.

**Intuition.** Two spins with correlation $c$ form a binary symmetric channel of crossover
$(1 - c)/2$, so all of Problem L1.4 applies verbatim.

**Solution.**

*Step 1 — build the joint.* Both marginals are uniform, and
$\Pr[\sigma_1 = \sigma_4] = \tfrac{1 + c}{2}$ with $c = t^{3}$, so the pair is a binary symmetric
channel with crossover $\epsilon = \tfrac{1 - c}{2}$.

*Step 2 — evaluate at $J / k_B T = 1$.* Here $t = \tanh 1 = 0.761594$, so $c = t^3 = 0.441744$ and
$\epsilon = 0.279128$.

*Step 3 — the mutual information.* By Problem L1.4,

$$
I(\sigma_1; \sigma_4) = 1 - H_b(0.279128) = 1 - 0.854260 = 0.145740 \ \text{bits}.
$$

*Step 4 — the two decay lengths.* Expanding $1 - H_b\!\left( \tfrac{1-c}{2} \right)$ for small $c$
gives $I \approx \frac{c^{2}}{2 \ln 2}$ bits, so $I \sim t^{2r}$ while
$\langle \sigma_1 \sigma_{1+r} \rangle \sim t^{r}$. With the correlation length
$\xi = -1/\ln t = 3.671861$ lattice spacings, the information decays with length

$$
\xi_I = \frac{\xi}{2} = 1.835930 .
$$

$$
\boxed{I(\sigma_1; \sigma_4) = 1 - H_b(0.279128) = 0.145740 \text{ bits}, \qquad \xi_I = \xi/2 = 1.835930}
$$

**Key takeaway.** Mutual information is quadratic in the correlation function at leading order, so
it dies twice as fast in space. Any diagnostic that looks for long-range order in the *information*
sees a shorter range than the correlation function does, which matters when mutual information is
used as an order parameter near a critical point.

In [23]:
for J_over_kT in (0.5, 1.0, 1.5):
    t = np.tanh(J_over_kT)
    print(f"  J/kT = {J_over_kT:.1f}:  t = tanh = {t:.6f},  xi = {-1 / np.log(t):.6f},"
          f"  xi_I = {-1 / (2 * np.log(t)):.6f}")
    for r in (1, 3):
        c = t ** r
        eps = (1.0 - c) / 2.0
        P_spin = np.array([[(1 + c) / 4, (1 - c) / 4], [(1 - c) / 4, (1 + c) / 4]])
        I_spin = mutual_information_bits(P_spin)
        print(f"      r = {r}:  c = {c:.6f}  eps = {eps:.6f}  I = {I_spin:.6f} bits"
              f"   quadratic approximation {c ** 2 / (2 * np.log(2)):.6f}")
        assert abs(I_spin - (1.0 - binary_entropy(np.array(eps)))) < 1e-14
t1 = np.tanh(1.0)
assert abs(mutual_information_bits(np.array(
    [[(1 + t1 ** 3) / 4, (1 - t1 ** 3) / 4], [(1 - t1 ** 3) / 4, (1 + t1 ** 3) / 4]]))
    - 0.1457401768641684) < 1e-12

  J/kT = 0.5:  t = tanh = 0.462117,  xi = 1.295443,  xi_I = 0.647721
      r = 1:  c = 0.462117  eps = 0.268941  I = 0.160058 bits   quadratic approximation 0.154045
      r = 3:  c = 0.098686  eps = 0.450657  I = 0.007037 bits   quadratic approximation 0.007025
  J/kT = 1.0:  t = tanh = 0.761594,  xi = 3.671861,  xi_I = 1.835930
      r = 1:  c = 0.761594  eps = 0.119203  I = 0.472935 bits   quadratic approximation 0.418400
      r = 3:  c = 0.441744  eps = 0.279128  I = 0.145740 bits   quadratic approximation 0.140762
  J/kT = 1.5:  t = tanh = 0.905148,  xi = 10.034465,  xi_I = 5.017233
      r = 1:  c = 0.905148  eps = 0.047426  I = 0.724640 bits   quadratic approximation 0.590995
      r = 3:  c = 0.741582  eps = 0.129209  I = 0.444735 bits   quadratic approximation 0.396701


## L3 — Challenge Proofs

### Problem L3.1 — Sufficiency as the equality case of the DPI

**Statement.** Prove that for a Markov chain $X \to Y \to T$, equality $I(X; Y) = I(X; T)$ holds if
and only if $X \to T \to Y$ is also a Markov chain, and use this to characterize sufficient
statistics.

**Intuition.** The gap in the data-processing inequality is a conditional mutual information, and a
conditional mutual information vanishes exactly at conditional independence.

**Solution.**

*Step 1 — the two expansions.* Theorem 4.3 applied to $I(X; Y, T)$ in both orders gives

$$
I(X; Y) + I(X; T \mid Y) = I(X; Y, T) = I(X; T) + I(X; Y \mid T).
$$

*Step 2 — use the assumed chain.* $X \to Y \to T$ means $I(X; T \mid Y) = 0$, so

$$
I(X; Y) - I(X; T) = I(X; Y \mid T).
$$

*Step 3 — the equality condition.* The left side vanishes if and only if $I(X; Y \mid T) = 0$, which
by Theorem 4.2 means $X$ and $Y$ are conditionally independent given $T$ — that is,
$X \to T \to Y$.

*Step 4 — the statistical reading.* Let $Y$ be data drawn from $p_\theta$, let $X = \theta$ be a
random parameter and let $T = T(Y)$ be a statistic. The chain $\theta \to Y \to T$ always holds, and

$$
I(\theta; T) = I(\theta; Y) \iff \theta \to T \to Y \iff p(y \mid t, \theta) = p(y \mid t),
$$

which is the Fisher-Neyman criterion: $T$ is sufficient exactly when the conditional law of the data
given $T$ is parameter-free.

$$
\boxed{I(X; Y) - I(X; T) = I(X; Y \mid T) \ge 0, \qquad \text{equality} \iff T \text{ is sufficient}}
$$

**Key takeaway.** Sufficiency, the data-processing inequality and conditional independence are three
views of one identity. A lossless representation *is* a sufficient statistic, which is the target
that the information bottleneck relaxes.

In [24]:
p_in = np.array([0.5, 0.5])
W1 = bsc(0.1)
P_suff = np.einsum("x,xy,yt->xyt", p_in, W1, np.array([[0.0, 1.0], [1.0, 0.0]]))
P_lossy = np.einsum("x,xy,yt->xyt", p_in, W1, bsc(0.2))
for name, P3 in (("bijective T (sufficient)", P_suff), ("noisy T (not sufficient)", P_lossy)):
    I_XY = mutual_information_bits(P3.sum(axis=2))
    I_XT = mutual_information_bits(P3.sum(axis=1))
    gap = conditional_mi_bits(P3)
    print(f"  {name}: I(X;Y) = {I_XY:.10f}  I(X;T) = {I_XT:.10f}"
          f"  I(X;Y|T) = {gap:.10f}  residual {abs((I_XY - I_XT) - gap):.2e}")
    assert abs((I_XY - I_XT) - gap) < 1e-14
assert conditional_mi_bits(P_suff) < 1e-14
assert conditional_mi_bits(P_lossy) > 0.1

  bijective T (sufficient): I(X;Y) = 0.5310044064  I(X;T) = 0.5310044064  I(X;Y|T) = 0.0000000000  residual 0.00e+00
  noisy T (not sufficient): I(X;Y) = 0.5310044064  I(X;T) = 0.1732536275  I(X;Y|T) = 0.3577507789  residual 4.44e-16


### Problem L3.2 — Gaussian inputs maximize the AWGN mutual information

**Statement.** For the channel $Y = X + N$ with $N \sim \mathcal{N}(0, \sigma^2)$ independent of $X$
and $\mathbb{E}[X^2] \le P$, prove $C = \tfrac{1}{2}\log_2\left( 1 + P/\sigma^2 \right)$ bits per
use, including the maximum-entropy step and the exact constraint it needs.

**Intuition.** Only the output entropy is under our control, and the Gaussian is the entropy
maximizer at a fixed second moment.

**Solution.**

*Step 1 — reduce to the output entropy.* Independence and translation invariance of differential
entropy give

$$
I(X; Y) = h(Y) - h(Y \mid X) = h(Y) - h(N) = h(Y) - \tfrac{1}{2}\ln\left( 2\pi e \sigma^2 \right).
$$

*Step 2 — the constraint on $Y$.* With $\mathbb{E}[N] = 0$ and $X \perp N$,

$$
\mathbb{E}[Y^2] = \mathbb{E}[X^2] + \sigma^2 \le P + \sigma^2 .
$$

This is a **second-moment** bound, and that is exactly the hypothesis the next step needs.

*Step 3 — the maximum-entropy lemma.* Among densities with $\mathbb{E}_p[Y^2] \le v$, the centred
Gaussian $\phi_v$ maximizes differential entropy. Indeed

$$
0 \le D_{\mathrm{KL}}(p \parallel \phi_v) = -h(p) - \int p(y) \ln \phi_v(y)\, dy,
$$

and since $\ln \phi_v(y) = -\tfrac{1}{2}\ln(2\pi v) - \frac{y^2}{2v}$ is quadratic, the integral
depends on $p$ only through its second moment:

$$
-\int p \ln \phi_v = \tfrac{1}{2}\ln(2\pi v) + \frac{\mathbb{E}_p[Y^2]}{2v} \le \tfrac{1}{2}\ln(2\pi v) + \tfrac{1}{2} = h(\phi_v).
$$

Hence $h(p) \le h(\phi_v)$, with equality iff $p = \phi_v$.

*Why the constraint must be on the second moment.* If only $\operatorname{Var}(Y) \le v$ were
assumed, a density with variance $v$ and mean $\mu$ has $\mathbb{E}[Y^2] = v + \mu^2$, and the last
inequality fails for large $\mu$. The lemma survives a variance constraint only when
$\mathbb{E}[Y] = 0$ is imposed as well, and shifting to zero mean changes neither $h$ nor $I$.

*Step 4 — assemble.* With $v = P + \sigma^2$,

$$
I(X; Y) \le \tfrac{1}{2}\ln\left( 2\pi e (P + \sigma^2) \right) - \tfrac{1}{2}\ln\left( 2\pi e \sigma^2 \right) = \tfrac{1}{2}\ln\left( 1 + \frac{P}{\sigma^2} \right) \ \text{nats}.
$$

*Step 5 — achievability.* $X \sim \mathcal{N}(0, P)$ makes $Y \sim \mathcal{N}(0, P + \sigma^2)$ and
attains the bound.

$$
\boxed{C = \tfrac{1}{2}\log_2\left( 1 + \frac{P}{\sigma^{2}} \right) \text{ bits per use, attained by } X \sim \mathcal{N}(0, P)}
$$

**Key takeaway.** "Gaussian is the worst noise and the best signal" follows from one lemma —
maximum entropy under a second-moment constraint — which recurs in rate-distortion theory and in the
Gaussian information bottleneck. State the constraint correctly or the lemma is false.

In [25]:
def diff_entropy_hist(samples, bins=400):
    """Differential entropy in nats from a histogram, for a sanity check only."""
    counts, edges = np.histogram(samples, bins=bins, density=True)
    width = edges[1] - edges[0]
    nz = counts > 0.0
    return float(-(counts[nz] * np.log(counts[nz]) * width).sum())


P_pow, sigma2, n_s = 3.0, 1.0, 400_000
noise = np.sqrt(sigma2) * rng.standard_normal(n_s)
print(f"  exact C = 0.5 ln(1 + P/sigma^2) = {0.5 * np.log(1 + P_pow / sigma2):.6f} nats"
      f" = {0.5 * np.log2(1 + P_pow / sigma2):.6f} bits")
inputs = (("Gaussian input", np.sqrt(P_pow) * rng.standard_normal(n_s)),
          ("uniform input", np.sqrt(3 * P_pow) * (rng.random(n_s) - 0.5) * 2),
          ("binary input", np.sqrt(P_pow) * rng.choice([-1.0, 1.0], n_s)))
estimates = {}
for name, x in inputs:
    y = x + noise
    estimates[name] = diff_entropy_hist(y) - 0.5 * np.log(2 * np.pi * np.e * sigma2)
    print(f"  {name:15s}: E[X^2] = {np.mean(x ** 2):.4f},"
          f"  h(Y) - h(N) = {estimates[name]:.6f} nats")
print("  the Gaussian input is the largest of the three, as Step 5 requires")
assert abs(0.5 * np.log2(1 + P_pow / sigma2) - 1.0) < 1e-12
assert estimates["Gaussian input"] > estimates["uniform input"] > estimates["binary input"]

  exact C = 0.5 ln(1 + P/sigma^2) = 0.693147 nats = 1.000000 bits
  Gaussian input : E[X^2] = 2.9890,  h(Y) - h(N) = 0.691173 nats
  uniform input  : E[X^2] = 2.9930,  h(Y) - h(N) = 0.673380 nats
  binary input   : E[X^2] = 3.0000,  h(Y) - h(N) = 0.585331 nats
  the Gaussian input is the largest of the three, as Step 5 requires


### Problem L3.3 — The Barber-Agakov bound and why every sample-based bound is capped

**Statement.** (a) Prove the variational lower bound
$I(X; Y) \ge H(X) - \mathbb{E}_{p(x,y)}\left[ -\log q_\phi(x \mid y) \right]$ for any decoder
$q_\phi$. (b) Explain why a lower bound computed from $K$ samples is limited to about $\log K$ nats.

**Intuition.** Any decoder gives a valid bound because the slack is its own KL error; and no
$K$-sample statistic can witness a density ratio much larger than $K$.

**Solution.**

*(a) Step 1 — start from the entropy form.*

$$
I(X; Y) = H(X) - H(X \mid Y) = H(X) + \mathbb{E}_{p(x,y)}\left[ \log p(x \mid y) \right].
$$

*(a) Step 2 — insert the decoder.*

$$
\mathbb{E}_{p(x,y)}\left[ \log p(x \mid y) \right]
= \mathbb{E}_{p(x,y)}\left[ \log q_\phi(x \mid y) \right]
+ \mathbb{E}_{p(y)}\left[ D_{\mathrm{KL}}\left( p(x \mid y) \parallel q_\phi(x \mid y) \right) \right].
$$

*(a) Step 3 — drop the nonnegative term.* Theorem 4.2 makes the divergence nonnegative, so

$$
I(X; Y) \ge H(X) + \mathbb{E}_{p(x,y)}\left[ \log q_\phi(x \mid y) \right] = H(X) - \mathcal{L}_{\text{recon}},
$$

with equality exactly when $q_\phi(x \mid y) = p(x \mid y)$. The bound is trainable: minimize a
reconstruction cross-entropy and the slack is the decoder's own KL error.

*(b) Step 4 — the sample ceiling.* A lower bound computed from $K$ samples is a statistic
$\hat{I}(x_{1:K}, y_{1:K})$ required to satisfy $\hat{I} \le I$ with high probability for *every*
joint law, including the independent one where $I = 0$. McAllester and Stratos construct pairs of
distributions — one with $I$ large and one independent — whose $K$-sample laws are
indistinguishable unless the sample exhibits an event of probability $O(e^{-I})$. With $K$ samples
the chance of seeing such evidence is $O\!\left( K e^{-I} \right)$, negligible once
$I \gg \log K$. No high-confidence bound can therefore certify much beyond $\log K$ nats, and the
ceiling of Theorem 4.8 is not an artefact of the InfoNCE derivation but the limit of the sampling
problem.

$$
\boxed{I(X; Y) \ge H(X) - \mathcal{L}_{\text{recon}}; \qquad \text{any } K\text{-sample lower bound} \lesssim \log K}
$$

**Key takeaway.** Lower-bounding mutual information is easy; *tightly* lower-bounding it from few
samples is impossible. "We maximize MI" should always be read as "we optimize a surrogate", never as
"we measured MI".

**Reference.** McAllester, D. and Stratos, K. "Formal limitations on the measurement of mutual
information", *AISTATS* (2020), Theorem 2.

In [26]:
P_ba = np.array([[0.35, 0.05], [0.10, 0.50]])
px_ba = P_ba.sum(axis=1)
py_ba = P_ba.sum(axis=0)
H_X_ba = entropy_bits(px_ba)
I_ba = mutual_information_bits(P_ba)
q_true = P_ba / py_ba[None, :]
for name, q in (("true posterior", q_true),
                ("marginal decoder", np.tile(px_ba[:, None], (1, 2))),
                ("uniform decoder", np.full((2, 2), 0.5))):
    recon = float(-(P_ba * np.log2(q)).sum())
    print(f"  {name:17s}: L_recon = {recon:.6f} bits,  bound = {H_X_ba - recon:+.6f},"
          f"  true I = {I_ba:.6f}")
    assert H_X_ba - recon <= I_ba + 1e-14
assert abs((H_X_ba - float(-(P_ba * np.log2(q_true)).sum())) - I_ba) < 1e-14

  true posterior   : L_recon = 0.585615 bits,  bound = +0.385335,  true I = 0.385335
  marginal decoder : L_recon = 0.970951 bits,  bound = +0.000000,  true I = 0.385335
  uniform decoder  : L_recon = 1.000000 bits,  bound = -0.029049,  true I = 0.385335


### Problem L3.4 — Fano's inequality applied to a classifier audit

**Statement.** A $1000$-class task has a uniform label prior and a deployed model reports $92\%$
top-1 accuracy. Use Theorem 4.7 to compute the minimum mutual information between features and label
that this accuracy implies, then run the argument in reverse: if an audit measured
$I(X; Y) = 5$ bits, what accuracy would be possible?

**Intuition.** Fano converts accuracy into a bound on residual entropy, and residual entropy into
bits; both directions are the same inequality read differently.

**Solution.**

*Step 1 — apply Fano.* With $K = 1000$ classes, $P_e = 0.08$ and $\hat{Y} = g(X)$,

$$
H(Y \mid X) \le H_b(P_e) + P_e \log_2 (K - 1).
$$

*Step 2 — evaluate the right side.*

$$
H_b(0.08) = 0.402179, \qquad 0.08 \log_2 999 = 0.797147, \qquad H(Y \mid X) \le 1.199326 \ \text{bits}.
$$

*Step 3 — convert to mutual information.* With $H(Y) = \log_2 1000 = 9.965784$ bits,

$$
I(X; Y) = H(Y) - H(Y \mid X) \ge 9.965784 - 1.199326 = 8.766458 \ \text{bits}.
$$

*Step 4 — the reverse direction.* If instead $I(X; Y) = 5$ bits, then $H(Y \mid X) = 4.965784$ bits
and the accuracy ceiling is the root of

$$
H_b(P_e) + P_e \log_2 999 = 4.965784 .
$$

The left side increases on $[0, 0.75]$, so the root is unique there; solving numerically gives
$P_e \ge 0.400863$, that is an accuracy ceiling of $59.91\%$. A reported $92\%$ on features carrying
only $5$ bits would therefore prove leakage or a broken evaluation.

$$
\boxed{I(X; Y) \ge \log_2 1000 - H_b(0.08) - 0.08\log_2 999 = 8.766458 \text{ bits}; \quad I = 5 \text{ bits} \implies \text{accuracy} \le 59.91\%}
$$

**Key takeaway.** Fano converts accuracy into bits and bits into an accuracy ceiling. Solve the
inequality numerically rather than guessing: at $P_e = 0.44$ the left side is $5.373898$, well past
the constraint, and quoting $0.44$ instead of $0.400863$ moves the reported ceiling by four
percentage points.

In [27]:
K_cls, acc = 1000, 0.92
Pe = 1.0 - acc
H_Y = np.log2(K_cls)
fano_rhs = binary_entropy(np.array(Pe)) + Pe * np.log2(K_cls - 1)
print(f"  H_b(0.08) = {binary_entropy(np.array(Pe)):.6f} bits")
print(f"  Pe log2 999 = {Pe * np.log2(K_cls - 1):.6f} bits")
print(f"  H(Y|X) <= {fano_rhs:.6f} bits;  H(Y) = {H_Y:.6f} bits")
print(f"  I(X;Y) >= {H_Y - fano_rhs:.6f} bits, i.e. {100 * (H_Y - fano_rhs) / H_Y:.1f}% of the label entropy")

target = H_Y - 5.0
root = brentq(lambda p: binary_entropy(np.array(p)) + p * np.log2(K_cls - 1) - target,
              0.0, 0.75, xtol=1e-14)
print(f"  with I = 5 bits: H(Y|X) = {target:.6f}, Fano root Pe = {root:.6f},"
      f" accuracy ceiling {100 * (1 - root):.2f}%")
print(f"  the often-quoted 0.44 gives H_b + Pe log2 999 = "
      f"{binary_entropy(np.array(0.44)) + 0.44 * np.log2(999):.6f}, far above {target:.6f}")
assert abs((H_Y - fano_rhs) - 8.76645782503642) < 1e-12
assert abs(root - 0.40086257627912836) < 1e-9

  H_b(0.08) = 0.402179 bits
  Pe log2 999 = 0.797147 bits
  H(Y|X) <= 1.199326 bits;  H(Y) = 9.965784 bits
  I(X;Y) >= 8.766458 bits, i.e. 88.0% of the label entropy
  with I = 5 bits: H(Y|X) = 4.965784, Fano root Pe = 0.400863, accuracy ceiling 59.91%
  the often-quoted 0.44 gives H_b + Pe log2 999 = 5.373898, far above 4.965784


### Problem L3.5 — The converse to the channel coding theorem

**Statement.** Let a $(2^{nR}, n)$ code be used over a discrete memoryless channel of capacity $C$,
with a uniform message $W$ and decoder $\hat{W} = g(Y^n)$. Prove
$P_e^{(n)} \ge 1 - C/R - 1/(nR)$, and deduce that $P_e^{(n)} \to 0$ forces $R \le C$. Evaluate the
bound for a binary symmetric channel with $\epsilon = 0.11$ at $R = 0.6$ and $n = 1000$.

**Intuition.** The message entropy has to be carried by the channel, and the channel carries at most
$nC$ bits; whatever is left over shows up as decoding error through Fano.

**Solution.**

*Step 1 — the single-letter bound.* Memorylessness gives
$H(Y^n \mid X^n) = \sum_i H(Y_i \mid X_i)$, and subadditivity gives
$H(Y^n) \le \sum_i H(Y_i)$. Subtracting,

$$
I(X^n; Y^n) \le \sum_{i=1}^{n} I(X_i; Y_i) \le n C .
$$

*Step 2 — split the message entropy.* $W$ uniform gives $H(W) = nR$ bits, and Theorem 4.1 gives
$nR = I(W; Y^n) + H(W \mid Y^n)$.

*Step 3 — bound the information term.* $W \to X^n \to Y^n$ is a Markov chain, so Theorem 4.4 and
Step 1 give $I(W; Y^n) \le I(X^n; Y^n) \le nC$.

*Step 4 — bound the residual with Fano.* $W \to Y^n \to \hat{W}$ is a Markov chain, so
$H(W \mid Y^n) \le H(W \mid \hat{W})$, and Theorem 4.7 with alphabet size $2^{nR}$ gives

$$
H(W \mid Y^n) \le H_b\!\left( P_e^{(n)} \right) + P_e^{(n)} \log_2\left( 2^{nR} - 1 \right) \le 1 + P_e^{(n)} n R .
$$

*Step 5 — combine and divide.* $nR \le nC + 1 + P_e^{(n)} nR$, hence

$$
P_e^{(n)} \ge 1 - \frac{C}{R} - \frac{1}{nR}.
$$

Letting $n \to \infty$ with $P_e^{(n)} \to 0$ gives $0 \ge 1 - C/R$, that is $R \le C$.

*Step 6 — the numbers.* $C = 1 - H_b(0.11) = 0.500084$ bits per use, so at $R = 0.6$ and
$n = 1000$,

$$
P_e^{(1000)} \ge 1 - \frac{0.500084}{0.6} - \frac{1}{600} = 0.164860,
$$

rising to $1 - C/R = 0.166527$ as $n \to \infty$.

$$
\boxed{P_e^{(n)} \ge 1 - \frac{C}{R} - \frac{1}{nR}; \qquad P_e^{(n)} \to 0 \implies R \le C}
$$

**Key takeaway.** Above capacity the error floor is a rate phenomenon, not a block-length one: the
only $n$-dependent term is $1/(nR)$, worth $0.001667$ here. Longer codes do not help; a lower rate
does.

In [28]:
C_bsc = 1.0 - binary_entropy(np.array(0.11))
R = 0.6
print(f"  C = 1 - H_b(0.11) = {C_bsc:.6f} bits per use;  R = {R}")
for n in (10, 100, 1000, 10000):
    floor = 1.0 - C_bsc / R - 1.0 / (n * R)
    print(f"  n = {n:6d}:  Pe >= {floor:.6f}   (block-length term {1.0 / (n * R):.6f})")
print(f"  limit 1 - C/R = {1.0 - C_bsc / R:.6f}")
for R_try in (0.4, 0.5, 0.6):
    print(f"  R = {R_try}:  1 - C/R = {1.0 - C_bsc / R_try:+.6f}"
          f"   {'above capacity, floor is positive' if R_try > C_bsc else 'below capacity, no floor'}")
assert abs((1.0 - C_bsc / R - 1.0 / 600.0) - 0.16485993027421333) < 1e-12

  C = 1 - H_b(0.11) = 0.500084 bits per use;  R = 0.6
  n =     10:  Pe >= -0.000140   (block-length term 0.166667)
  n =    100:  Pe >= 0.149860   (block-length term 0.016667)
  n =   1000:  Pe >= 0.164860   (block-length term 0.001667)
  n =  10000:  Pe >= 0.166360   (block-length term 0.000167)
  limit 1 - C/R = 0.166527
  R = 0.4:  1 - C/R = -0.250210   below capacity, no floor
  R = 0.5:  1 - C/R = -0.000168   below capacity, no floor
  R = 0.6:  1 - C/R = +0.166527   above capacity, floor is positive


### Problem L3.6 — Concavity and the KKT conditions for capacity

**Statement.** Using Theorem 4.5, prove that an input distribution $p^{\star}$ achieves capacity if
and only if

$$
D_{\mathrm{KL}}\left( p(y \mid x) \parallel p^{\star}_Y \right) = C \ \text{ for every } x \text{ with } p^{\star}(x) \gt 0,
\qquad \le C \ \text{ otherwise},
$$

where $p^{\star}_Y$ is the output law induced by $p^{\star}$. Verify it on the Z-channel of
Section 7.4.

**Intuition.** At an interior optimum every used input must be equally valuable, or shifting mass
towards the better one would raise $I$.

**Solution.**

*Step 1 — write $I$ as an average of divergences.* For a fixed channel,

$$
I(p) = \sum_x p(x)\, D_{\mathrm{KL}}\left( p(y \mid x) \parallel p_Y \right), \qquad p_Y = \sum_x p(x) p(y \mid x).
$$

*Step 2 — differentiate.* Work in nats and write
$D_x(p) = D_{\mathrm{KL}}\left( p(\cdot \mid x) \parallel p_Y \right)$. Differentiating $I(p)$ with
respect to $p(x)$ produces one term from the explicit factor $p(x)$ and one from the dependence of
$p_Y$ on $p(x)$:

$$
\frac{\partial I}{\partial p(x)} = D_x(p) \; - \; \sum_{x'} p(x') \sum_y p(y \mid x') \, \frac{p(y \mid x)}{p_Y(y)} .
$$

The second term collapses, because the inner sum over $x'$ rebuilds $p_Y(y)$:

$$
\sum_y \frac{p(y \mid x)}{p_Y(y)} \sum_{x'} p(x')\, p(y \mid x') = \sum_y \frac{p(y \mid x)}{p_Y(y)} \, p_Y(y) = \sum_y p(y \mid x) = 1 .
$$

Hence

$$
\frac{\partial I}{\partial p(x)} = D_x(p) - 1 \quad \text{in nats} .
$$

*Step 3 — apply the first-order conditions.* By Theorem 4.5 the map $p \mapsto I(p)$ is concave on
the simplex, so the Karush-Kuhn-Tucker conditions of a concave maximization over
$\lbrace p \ge 0, \sum_x p(x) = 1 \rbrace$ are necessary **and** sufficient. With multiplier $\nu$
for the equality constraint they read

$$
D_x(p^{\star}) - 1 = \nu \quad \text{when } p^{\star}(x) \gt 0, \qquad
D_x(p^{\star}) - 1 \le \nu \quad \text{when } p^{\star}(x) = 0 .
$$

*Step 4 — identify the constant.* Multiply the active equalities by $p^{\star}(x)$ and sum: the left
side is $I(p^{\star}) - 1 = C - 1$ and the right side is $\nu$, so $\nu = C - 1$ and the conditions
become $D_x(p^{\star}) = C$ on the support and $\le C$ off it.

$$
\boxed{p^{\star} \text{ achieves capacity} \iff D_{\mathrm{KL}}\left( p(y \mid x) \parallel p^{\star}_Y \right) = C \text{ on the support}, \ \le C \text{ off it}}
$$

**Key takeaway.** Concavity is what upgrades a first-order condition into a certificate: any $p$
satisfying these equalities *is* optimal, with no need to check second-order conditions or to worry
about local maxima. This is also the stopping test for the Blahut-Arimoto iteration.

In [29]:
def blahut_arimoto(channel, tol=1e-14, max_iter=20000):
    r = np.full(channel.shape[0], 1.0 / channel.shape[0])
    for _ in range(max_iter):
        qy = r @ channel
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = np.where(channel > 0.0,
                             np.log(np.where(channel > 0.0, channel, 1.0))
                             - np.log(np.where(qy > 0.0, qy, 1.0))[None, :], 0.0)
        r_new = r * np.exp((channel * ratio).sum(axis=1))
        r_new /= r_new.sum()
        if np.max(np.abs(r_new - r)) < tol:
            r = r_new
            break
        r = r_new
    return mutual_information_bits(r[:, None] * channel), r


def kl_rows_bits(channel, qy):
    out = np.empty(channel.shape[0])
    for x in range(channel.shape[0]):
        row = channel[x]
        nz = row > 0.0
        out[x] = float((row[nz] * np.log2(row[nz] / qy[nz])).sum())
    return out


for name, W_ch in (("BSC(0.2)", bsc(0.2)),
                   ("Z-channel", np.array([[1.0, 0.0], [0.25, 0.75]])),
                   ("useless input added", np.array([[1.0, 0.0], [0.25, 0.75], [0.6, 0.4]]))):
    C_ba, r_ba = blahut_arimoto(W_ch)
    qy = r_ba @ W_ch
    D = kl_rows_bits(W_ch, qy)
    print(f"  {name:20s} C = {C_ba:.10f} bits   input {np.round(r_ba, 6)}")
    print(f"      D(p(.|x) || p*_Y) per input: {np.round(D, 10)}")
    for x in range(W_ch.shape[0]):
        if r_ba[x] > 1e-8:
            assert abs(D[x] - C_ba) < 1e-8
        else:
            assert D[x] <= C_ba + 1e-8

  BSC(0.2)             C = 0.2780719051 bits   input [0.5 0.5]
      D(p(.|x) || p*_Y) per input: [0.2781 0.2781]
  Z-channel            C = 0.5582386267 bits   input [0.5722 0.4278]
      D(p(.|x) || p*_Y) per input: [0.5582 0.5582]
  useless input added  C = 0.5582386267 bits   input [0.5722 0.4278 0.    ]
      D(p(.|x) || p*_Y) per input: [0.5582 0.5582 0.02  ]
